## Phase 1 Downscaling: IDW

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin
from rasterio.warp import transform, transform_bounds
from scipy.spatial import cKDTree

#### Manually Enter Sensor Coordinates

In [20]:
SENSOR_COORDS = {
    "cubao-1HR-clean": (14.6173, 121.0593),
    "lawton-1HR-clean": (14.5946, 120.9794),
    "makati-1HR-clean": (14.5547, 121.0244),
    "pasay-1HR-clean": (14.5375, 121.0014),
    "SDG-1HR-clean": (14.3537, 120.5921),
}

In [19]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output/IDW")

#### IDW Paramaters

In [4]:
IDW_P = 2
SEARCH_RADIUS = 10_000 # 10 km 
RESOLUTION = 100 
NODATA = -9999.0

In [5]:
REQUIRE_MIN_SENSORS = True

In [6]:
# Boundary Box of Manila
BBOX_WGS84 = {
    "min_lon": 120.855,
    "max_lon": 121.205,
    "min_lat":  14.390,
    "max_lat":  14.810,
}

In [7]:
CRS_UTM   = CRS.from_epsg(32651)
CRS_WGS84 = CRS.from_epsg(4326)

In [8]:
BAND_DEFS = [
    ("pm1", "PM1 (ug/m3)"),
    ("pm25", "PM2.5 (ug/m3)"),
    ("pm10", "PM10 (ug/m3)"),
    ("temperature", "Temperature (degC)"),
    ("humidity", "Humidity (%)")
]

In [9]:
BANDS = [key for key, _ in BAND_DEFS]
BAND_NAMES = {index: name for index, (_, name) in enumerate(BAND_DEFS, start=1)}

In [10]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    clean_names = {
        column: re.sub(r"_+", "_", re.sub(r"[^a-zA-Z0-9]", "_", re.sub(r"\(.*?\)", "", column)))
        .strip("_")
        .lower()
        for column in df.columns
    }
    df = df.rename(columns=clean_names)

    aliases = {}
    for column in df.columns:
        if "timestamp" in column or column in {"time", "date", "datetime"}:
            aliases[column] = "timestamp"
        elif re.search(r"pm_?2_?5", column) or "pm25" in column:
            aliases[column] = "pm25"
        elif re.search(r"pm_?10", column) or "pm10" in column:
            aliases[column] = "pm10"
        elif re.search(r"\bpm_?1\b", column) or column in {"pm1", "pm_1"}:
            aliases[column] = "pm1"
        elif "temp" in column:
            aliases[column] = "temperature"
        elif "humid" in column:
            aliases[column] = "humidity"

    return df.rename(columns=aliases)

In [11]:
def load_sensor_csv(path: Path, lat: float, lon: float) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python", encoding_errors="replace")
    df = normalize_columns(df)

    if "timestamp" not in df.columns:
        raise ValueError(f"{path.name}: no timestamp column found. Columns: {df.columns.tolist()}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    for band in BANDS:
        if band not in df.columns:
            df[band] = np.nan

    hourly = df.groupby(df["timestamp"].dt.floor("h"), as_index=False)[BANDS].mean()
    hourly = hourly.rename(columns={"timestamp": "hour"})
    hourly["lat"] = lat
    hourly["lon"] = lon
    return hourly

In [12]:
def load_all_sensors(data_dir: Path) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(data_dir.glob("*.csv")):
        if csv_path.stem not in SENSOR_COORDS:
            print(f"Skipping {csv_path.name}: no coordinates configured")
            continue

        lat, lon = SENSOR_COORDS[csv_path.stem]
        sensor_df = load_sensor_csv(csv_path, lat, lon)
        sensor_df["sensor_id"] = csv_path.stem
        frames.append(sensor_df)
        print(f"{csv_path.name}: {len(sensor_df)} hourly records")

    if not frames:
        raise FileNotFoundError(f"No configured sensor CSVs found in {data_dir}")

    return pd.concat(frames, ignore_index=True)

### Create Grid

In [13]:
def build_grid(): 
    minx, miny, maxx, maxy = transform_bounds(
        CRS_WGS84,
        CRS_UTM,
        BBOX_WGS84["min_lon"],
        BBOX_WGS84["min_lat"],
        BBOX_WGS84["max_lon"],
        BBOX_WGS84["max_lat"],
    )

    minx = np.floor(minx / RESOLUTION) * RESOLUTION
    miny = np.floor(miny / RESOLUTION) * RESOLUTION
    maxx = np.ceil(maxx / RESOLUTION) * RESOLUTION
    maxy = np.ceil(maxy / RESOLUTION) * RESOLUTION

    cols = int(round((maxx - minx) / RESOLUTION))
    rows = int(round((maxy - miny) / RESOLUTION))
    transform_matrix = from_origin(minx, maxy, RESOLUTION, RESOLUTION)

    xs = minx + (np.arange(cols) + 0.5) * RESOLUTION
    ys = maxy - (np.arange(rows) + 0.5) * RESOLUTION
    grid_x, grid_y = np.meshgrid(xs, ys)

    meta = {
        "driver": "GTiff",
        "dtype": "float32",
        "width": cols,
        "height": rows,
        "count": len(BANDS),
        "crs": CRS_UTM,
        "transform": transform_matrix,
        "nodata": NODATA,
        "compress": "deflate",
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
    }

    grid_xy = np.column_stack((grid_x.ravel(), grid_y.ravel()))
    print(f"Grid: {rows} rows x {cols} cols ({rows * cols:,} cells)")
    return grid_xy, meta, rows, cols

In [14]:
def project_sensors(df: pd.DataFrame) -> pd.DataFrame:
    utm_x, utm_y = transform(CRS_WGS84, CRS_UTM, df["lon"].to_numpy(), df["lat"].to_numpy())
    projected = df.copy()
    projected["utm_x"] = utm_x
    projected["utm_y"] = utm_y
    return projected

In [15]:
def idw_surface(sensor_xy: np.ndarray, sensor_values: np.ndarray, grid_xy: np.ndarray) -> np.ndarray:
    sensor_count = len(sensor_xy)
    distances, indices = cKDTree(sensor_xy).query(
        grid_xy,
        k=sensor_count,
        distance_upper_bound=SEARCH_RADIUS,
        workers=-1,
    )

    distances = np.reshape(distances, (-1, sensor_count))
    indices = np.reshape(indices, (-1, sensor_count))

    in_radius = indices < sensor_count
    has_neighbour = in_radius.any(axis=1)
    safe_indices = np.where(in_radius, indices, 0)
    values = sensor_values[safe_indices]
    surface = np.full(len(grid_xy), np.nan, dtype=np.float32)

    exact = (distances == 0) & in_radius
    has_exact = exact.any(axis=1)
    if has_exact.any():
        exact_columns = np.argmax(exact[has_exact], axis=1)
        surface[has_exact] = values[has_exact][np.arange(has_exact.sum()), exact_columns]

    interpolate = has_neighbour & ~has_exact
    if interpolate.any():
        valid_distances = np.where(in_radius[interpolate], distances[interpolate], np.inf)
        valid_values = np.where(in_radius[interpolate], values[interpolate], 0.0)
        weights = np.where(np.isfinite(valid_distances), 1.0 / valid_distances**IDW_P, 0.0)
        weight_sum = weights.sum(axis=1)
        surface[interpolate] = (weights * valid_values).sum(axis=1) / weight_sum

    return surface

In [16]:
def write_geotiff(band_grids: dict[str, np.ndarray], rows: int, cols: int, meta: dict, out_path: Path):
    with rasterio.open(out_path, "w", **meta) as dst:
        for band_index, band in enumerate(BANDS, start=1):
            surface = band_grids[band].reshape(rows, cols).astype(np.float32)
            dst.write(np.where(np.isnan(surface), NODATA, surface), band_index)
            dst.set_band_description(band_index, BAND_NAMES[band_index])

            valid = surface[~np.isnan(surface)]
            if valid.size:
                dst.update_tags(
                    band_index,
                    STATISTICS_MINIMUM=f"{valid.min():.6f}",
                    STATISTICS_MAXIMUM=f"{valid.max():.6f}",
                    STATISTICS_MEAN=f"{valid.mean():.6f}",
                    STATISTICS_STDDEV=f"{valid.std():.6f}",
                    STATISTICS_VALID_PERCENT=f"{100 * valid.size / surface.size:.2f}",
                )

In [17]:
def run_pipeline():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    grid_xy, meta, rows, cols = build_grid()
    sensor_df = project_sensors(load_all_sensors(DATA_DIR))
    min_sensors = 2 if REQUIRE_MIN_SENSORS else 1

    written = 0
    skipped = 0

    for hour, hour_df in sensor_df.groupby("hour", sort=True):
        active = hour_df.dropna(subset=["pm25"])
        if len(active) < min_sensors:
            skipped += 1
            continue

        sensor_xy = active[["utm_x", "utm_y"]].to_numpy()
        band_grids = {}

        for band in BANDS:
            values = active[band].to_numpy(dtype=np.float32)
            valid = ~np.isnan(values)
            band_grids[band] = (
                idw_surface(sensor_xy[valid], values[valid], grid_xy)
                if valid.any()
                else np.full(len(grid_xy), np.nan, dtype=np.float32)
            )

        out_path = OUTPUT_DIR / f"{pd.Timestamp(hour).strftime('%Y-%m-%d_%H')}.tif"
        write_geotiff(band_grids, rows, cols, meta, out_path)
        written += 1

    print(f"GeoTIFFs written: {written}")
    print(f"Hours skipped: {skipped}")
    print(f"Output folder: {OUTPUT_DIR.resolve()}")

In [21]:
run_pipeline()

Grid: 469 rows x 382 cols (179,158 cells)
cubao-1HR-clean.csv: 2697 hourly records
lawton-1HR-clean.csv: 2738 hourly records
makati-1HR-clean.csv: 234 hourly records
pasay-1HR-clean.csv: 633 hourly records
SDG-1HR-clean.csv: 103 hourly records


KeyboardInterrupt: 